# Controlling MODFLOW 6 with the API — B. Monitoring convergence

This is part B of the MODFLOW 6 API series — see [**A. Basic usage**](modflow-api-A.ipynb) for the API lifecycle (`initialize → update`/`solve → finalize`) and the simulation → models → packages mental model.

Step into the solver and watch how the solution converges: after preparing each time step, loop over the solver's outer iterations by hand, read the maximum head change at every iteration, and refresh a live plot after each time step so you can watch convergence happen as the model runs.

Run the imports and library-location cells below first, then continue to the convergence monitor.

## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline
import os
import pathlib as pl
import platform

import flopy
import matplotlib.pyplot as plt
import numpy as np
from modflowapi import ModflowApi

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. Find the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the executable inside the active environment (here the project's pixi environment), and confirm that both exist.

In [ ]:
env_path = pl.Path(os.environ.get("CONDA_PREFIX", None))
assert env_path is not None, "Notebook must be run from the mf6xtd pixi environment"

bin_path = "bin"
exe_ext = ""
if "linux" in platform.platform().lower():
    lib_ext = ".so"
elif "darwin" in platform.platform().lower() or "macos" in platform.platform().lower():
    lib_ext = ".dylib"
else:
    bin_path = "Scripts"
    lib_ext = ".dll"
    exe_ext = ".exe"
lib_name = env_path / f"{bin_path}/libmf6{lib_ext}"
if not lib_name.is_file():
    lib_name = env_path / f"lib/libmf6{lib_ext}"
if not lib_name.is_file():
    raise FileNotFoundError(
        f"Could not find mf6 library at in either: {env_path / 'bin'}"
        + f" or {env_path / 'lib'}"
    )

mf6_exe = env_path / f"{bin_path}/mf6{exe_ext}"

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Monitor convergence

Step into the solver to watch how the solution converges. After preparing each time step, loop over the solver's outer iterations manually and read the maximum head change (`HNCG`) at every iteration. The `|maximum head change|` for every iteration is plotted on a log scale and the figure is refreshed after each time step, so you can watch convergence happen **as the model runs**. The x-axis is grouped by stress period and time step (read from the `TDIS` package, with labels thinned when there are many); each time step is given an **equal-width slot** so the lengthy initial steady-state solve does not crowd out the equally interesting transient time steps. The figure uses the USGS publication style from `flopy.plot.styles`. This exposes the inner workings that are normally hidden when MODFLOW 6 runs on its own, and is useful for diagnosing slow or failing convergence.

In [ ]:
ws = pl.Path(f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}")
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api04")

sim = flopy.mf6.MFSimulation.load(sim_name=name, sim_ws=ws, write_headers=False)
sim.memory_print_option = "all"
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
from IPython.display import clear_output, display

mf6 = ModflowApi(lib_name, working_directory=new_ws)
mf6.initialize()

# solver settings and the live pointer to the maximum head change per outer
# iteration (HNCG); KPER and KSTP are read from the TDIS package each time step
max_iter = int(mf6.get_value(mf6.get_var_address("MXITER", "SLN_1"))[0])
max_change = mf6.get_value_ptr(mf6.get_var_address("HNCG", "SLN_1"))
kper_tag = mf6.get_var_address("KPER", "TDIS")
kstp_tag = mf6.get_var_address("KSTP", "TDIS")

current_time = mf6.get_current_time()
end_time = mf6.get_end_time()

# convergence history, one entry per outer iteration:
# (cumulative iteration, |maximum head change|, stress period, time step)
history = []
not_converged = []
max_ticks = 25  # most SP/TS labels to show before thinning


def plot_convergence(fig, ax):
    ax.clear()
    if history:
        # group consecutive iterations into (stress period, time step) blocks
        blocks, b0 = [], 0
        for k in range(len(history)):
            if k == len(history) - 1 or history[k][2:] != history[k + 1][2:]:
                blocks.append((b0, k))
                b0 = k + 1

        # give every time step an equal-width slot on the x-axis so the long
        # initial steady-state solve (many outer iterations) does not crowd out
        # the transient time steps. Within slot bi (spanning [bi, bi + 1]) the
        # iterations are spread evenly, so each time step gets the same width
        # regardless of how many iterations it needed.
        xs, ys = [], []
        for bi, (i0, i1) in enumerate(blocks):
            n = i1 - i0 + 1
            for j, k in enumerate(range(i0, i1 + 1)):
                xs.append(bi + (j + 0.5) / n if n > 1 else bi + 0.5)
                yv = history[k][1]
                ys.append(np.nan if yv == 0.0 else yv)  # skip exact zeros on log axis
        ax.semilogy(
            xs,
            ys,
            marker="o",
            ms=4,
            lw=1.0,
            color="0.25",
            mfc="cyan",
            mec="0.25",
        )

        # thin the labels/separators so at most ~max_ticks are drawn
        stride = max(1, -(-len(blocks) // max_ticks))
        ticks, labels = [], []
        for bi, (i0, i1) in enumerate(blocks):
            if bi % stride == 0 or bi == len(blocks) - 1:
                ticks.append(bi + 0.5)
                labels.append(f"SP {history[i1][2]}\nTS {history[i1][3]}")
                if bi > 0:
                    ax.axvline(bi, color="0.5", lw=0.5)
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels)

    flopy.plot.styles.heading(ax=ax, heading="Outer iteration convergence")
    flopy.plot.styles.xlabel(ax=ax, label="Stress period and time step")
    flopy.plot.styles.ylabel(ax=ax, label="Maximum head change, ft")
    flopy.plot.styles.remove_edge_ticks(ax=ax)

    clear_output(wait=True)
    display(fig)


with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)

    niter = 0
    while current_time < end_time:
        dt = mf6.get_time_step()
        mf6.prepare_time_step(dt)
        kper = int(mf6.get_value(kper_tag)[0])
        kstp = int(mf6.get_value(kstp_tag)[0])

        mf6.prepare_solve()
        kiter = 0
        has_converged = False
        while kiter < max_iter:
            has_converged = mf6.solve()
            niter += 1
            history.append((niter, abs(float(max_change[kiter])), kper, kstp))
            kiter += 1
            if has_converged:
                break

        plot_convergence(fig, ax)  # live update once per time step

        if not has_converged:
            not_converged.append((kper, kstp))

        mf6.finalize_solve()
        mf6.finalize_time_step()
        current_time = mf6.get_current_time()

    mf6.finalize()

plt.close(fig)  # the live figure is already displayed; avoid a duplicate

if not_converged:
    print("did not converge:", ", ".join(f"SP {p} TS {t}" for p, t in not_converged))
else:
    print("all time steps converged")

**How to read this plot.** Each marker is one outer (Picard) iteration. Within a time step the curve drops as the head change shrinks toward convergence, then resets at the start of the next time step — the vertical lines and `SP`/`TS` labels mark those breaks. A time step that reaches a small head change in just a few iterations converged easily; a curve that stays high or runs all the way to the `MXITER` cap flags a time step that is hard to solve. This is exactly the diagnostic information MODFLOW normally hides when it runs on its own.